In [1]:
import os
import urllib.request
import gzip
import shutil

import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri

import ipywidgets as widgets
from IPython.display import display, clear_output


# reload(module) is required for developing:
# this way one doesn't have to restart kernel at any change in the package
from importlib import reload

from utils import textutils as tut
tut = reload(tut)

from utils import magutils as mut
mut = reload(mut)

from utils import catutils as catut
catut = reload(catut)

from widgets import save_panel as sp_module
sp_module = reload(sp_module)

from widgets import category_panel as cp_module
cp_module = reload(cp_module)


from widgets import plot_panel as pp_module
pp_module = reload(pp_module)

from utils import plotutils as put
put = reload(put)

In [2]:
## checks if the dataset exists. If not -> download

DATA_URL = "https://zenodo.org/records/15363836/files/structures_fe_share.pckl.gzip?download=1"
GZIP_FILENAME = "structures_fe_share.pckl.gzip"
OUT_FILENAME = "structures_fe_share.pckl"

if not os.path.exists(OUT_FILENAME):
    if not os.path.exists(GZIP_FILENAME):
        print(f"{GZIP_FILENAME} not found. Downloading from Zenodo...")
        urllib.request.urlretrieve(DATA_URL, GZIP_FILENAME)
        print("Download complete.")

    print(f"Decompressing {GZIP_FILENAME}...")
    with gzip.open(GZIP_FILENAME, 'rb') as f_in:
        with open(OUT_FILENAME, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print(f"Saved decompressed file as {OUT_FILENAME}")

    os.remove(GZIP_FILENAME)
    print(f"Removed compressed file {GZIP_FILENAME}")
else:
    print(f"{OUT_FILENAME} already exists.")

structures_fe_share.pckl already exists.


In [3]:
df = pd.read_pickle("structures_fe_share.pckl")

In [4]:
df.head()

,name,energy,forces,energy_corrected,energy_corrected_per_atom,ase_atoms,mag_mom
0,~/VASP_test/large_volumes/bcc/free_6/0.005,-0.435437,"[[0.0, 0.0, 0.0]]",-0.000022,-0.000022,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.0, -0.001, 0.039]]"
1,~/VASP_test/large_volumes/bcc/free_6/0.100,-0.442888,"[[0.0, 0.0, 0.0]]",-0.007473,-0.007473,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[-0.0, 0.0, 0.226]]"
2,~/VASP_test/large_volumes/bcc/free_6/0.150,-0.577722,"[[0.0, 0.0, 0.0]]",-0.142307,-0.142307,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[-0.0, 0.0, 0.493]]"
3,~/VASP_test/large_volumes/bcc/free_6/0.750,-1.114440,"[[0.0, 0.0, 0.0]]",-0.679025,-0.679025,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.0, -0.0, 1.194]]"
4,~/VASP_test/large_volumes/bcc/free_6/0.850,-0.975302,"[[0.0, 0.0, 0.0]]",-0.539887,-0.539887,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.002, 0.001, 1.065]]"


# 1.Data Preprocessing

## 1.1. Enrich dataset with per-atom and aggregate magnetic/structural properties

In [5]:
df['nat'] = df['ase_atoms'].map(mut.get_n_atoms)
df['volume'] = df['ase_atoms'].map(mut.get_volume)
df['volume_per_atom'] = df['volume'] / df['nat']

df['M'] = df['mag_mom'].map(mut.total_mag_moment)
df['M_per_atom'] = df['M'] / df['nat']
df['M_abs'] = df['mag_mom'].map(mut.total_mag_moment_abs)
df['M_abs_per_atom'] = df['M_abs'] / df['nat']

df['force_max'] = df['forces'].map(mut.max_force)
df['mag_mom_max'] = df['mag_mom'].map(mut.max_mag_mom)

df["all_mag_mom_eq"] = df["mag_mom"].map(mut.all_moments_equal)
df['if_coll'] = df['mag_mom'].map(mut.is_collinear)

### 1.1.1. Check for None/Nan values

In [6]:
nan_locs = df.isna()
if nan_locs.any().any():
    print("⚠️ Found NaNs in the DataFrame:")
    for col in df.columns:
        n_missing = nan_locs[col].sum()
        if n_missing > 0:
            print(f"  - {col}: {n_missing} missing values: {df[col][df[col].isna()].index.tolist()}")
else:
    print("✅ No NaNs found.")

⚠️ Found NaNs in the DataFrame:
  - volume: 15 missing values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
  - volume_per_atom: 15 missing values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]


In [7]:
display(df[df.isna().any(axis=1)])

,name,energy,forces,energy_corrected,energy_corrected_per_atom,ase_atoms,mag_mom,nat,volume,volume_per_atom,M,M_per_atom,M_abs,M_abs_per_atom,force_max,mag_mom_max,all_mag_mom_eq,if_coll
0,~/VASP_test/large_volumes/bcc/free_6/0.005,-0.435437,"[[0.0, 0.0, 0.0]]",-0.000022,-0.000022,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.0, -0.001, 0.039]]",1,NaN,NaN,0.039013,0.039013,0.039013,0.039013,0.0,0.039013,True,True
1,~/VASP_test/large_volumes/bcc/free_6/0.100,-0.442888,"[[0.0, 0.0, 0.0]]",-0.007473,-0.007473,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[-0.0, 0.0, 0.226]]",1,NaN,NaN,0.226000,0.226000,0.226000,0.226000,0.0,0.226000,True,True
2,~/VASP_test/large_volumes/bcc/free_6/0.150,-0.577722,"[[0.0, 0.0, 0.0]]",-0.142307,-0.142307,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[-0.0, 0.0, 0.493]]",1,NaN,NaN,0.493000,0.493000,0.493000,0.493000,0.0,0.493000,True,True
3,~/VASP_test/large_volumes/bcc/free_6/0.750,-1.114440,"[[0.0, 0.0, 0.0]]",-0.679025,-0.679025,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.0, -0.0, 1.194]]",1,NaN,NaN,1.194000,1.194000,1.194000,1.194000,0.0,1.194000,True,True
4,~/VASP_test/large_volumes/bcc/free_6/0.850,-0.975302,"[[0.0, 0.0, 0.0]]",-0.539887,-0.539887,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.002, 0.001, 1.065]]",1,NaN,NaN,1.065002,1.065002,1.065002,1.065002,0.0,1.065002,True,True
5,~/VASP_test/large_volumes/bcc/free_6/1.000,-1.622238,"[[0.0, 0.0, 0.0]]",-1.186823,-1.186823,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.001, -0.0, 1.726]]",1,NaN,NaN,1.726000,1.726000,1.726000,1.726000,0.0,1.726000,True,True
6,~/VASP_test/large_volumes/bcc/free_6/1.250,-1.837182,"[[0.0, 0.0, 0.0]]",-1.401767,-1.401767,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.002, -0.001, 1.934]]",1,NaN,NaN,1.934001,1.934001,1.934001,1.934001,0.0,1.934001,True,True
7,~/VASP_test/large_volumes/bcc/free_6/1.350,-1.989831,"[[0.0, 0.0, 0.0]]",-1.554416,-1.554416,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[-0.0, -0.0, 2.095]]",1,NaN,NaN,2.095000,2.095000,2.095000,2.095000,0.0,2.095000,True,True
8,~/VASP_test/large_volumes/bcc/free_6/1.450,-2.174441,"[[0.0, 0.0, 0.0]]",-1.739026,-1.739026,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.0, 0.0, 2.31]]",1,NaN,NaN,2.310000,2.310000,2.310000,2.310000,0.0,2.310000,True,True
9,~/VASP_test/large_volumes/bcc/free_6/2.250,-2.892608,"[[0.0, 0.0, 0.0]]",-2.457193,-2.457193,"(Atom('Fe', [np.float64(0.0), np.float64(0.0),...","[[0.0, 0.0, 3.292]]",1,NaN,NaN,3.292000,3.292000,3.292000,3.292000,0.0,3.292000,True,True


### 1.1.2. Number of structures with collinear - non collinear margnetic vectors

In [8]:
df['if_coll'].value_counts()

if_coll
True     61299
False     7695
Name: count, dtype: int64

## 1.2. Categorization based on the "name" column values

In [9]:
# For example:
#   - name: "~/VASP_test/large_volumes/bcc/free_6/0.005"
#     → category: "bcc"
#   - name: "/storage/rinalm9z/c_a_AFM/0.8/0.2/2.3773939154..."
#     → category: "c_a"
#
# Categories are assigned as additional boolean columns in the dataframe (multi-label encoding - i.e. one row can belong to multiple categories)
# An extra column "no_cat" marks rows that match none of the defined categories for an extra checkup.

In [10]:
categories_patterns = {
    "bcc": re.compile(r"bcc"),
    "fcc": re.compile(r"fcc"),
    "supercell": re.compile(r"supercell"),
    "phonons": re.compile("phonon|frozen"),
    "bain_path": re.compile("c_a"),
    "hcp": re.compile("hcp"),
    "surface": re.compile("surf"),
    "gamma": re.compile("gamma"),
    "vacancy": re.compile("vacanc"),
    "green_boundary": re.compile(r"S\d+_\d+"),
    "defect": re.compile("defect"),
    "rotation": re.compile("rotat"),
}

# ig
catut.categorize_in_place(df, categories_patterns)

categories = (*categories_patterns.keys(), 'no_cat') # add 'no category' category as an extra checkup (should be zero)

catut.category_count(df, categories)


{'bcc': 29446,
 'fcc': 4198,
 'supercell': 12155,
 'phonons': 2613,
 'bain_path': 13210,
 'hcp': 1706,
 'surface': 1258,
 'gamma': 4340,
 'vacancy': 3198,
 'green_boundary': 975,
 'defect': 11917,
 'rotation': 5226,
 'no_cat': 0}

# 2. Category Panel

## 2.1. TargetFeature: What and Why

In datasets, some columns (like `forces` or `mag_mom`) contain **vector-valued data** (e.g. per-atom 3D vectors), while others are **scalar** (e.g. energy per atom).

To unify their handling — especially for plotting histograms or computing statistics — we use `TargetFeature`.

Each `TargetFeature` has:
- `name`: column name in the DataFrame
- `extractor`: a function that converts the Series into a 1D array of values (e.g. norms of vectors)
- `is_atom_level`: whether this is atom-wise data (optional)

If `extractor=None`, the values are taken “as-is” via `.to_numpy()`.

#### Example:

```python
features = [
    TargetFeature("energy_corrected_per_atom"),  # scalar
    TargetFeature("forces", lambda s: np.linalg.norm(np.vstack(s), axis=1), is_atom_level=True),
    TargetFeature("mag_mom", lambda s: np.linalg.norm(np.vstack(s), axis=1), is_atom_level=True),
    TargetFeature("nat")  # scalar
]
```
These features are passed to a Panel instance (e.g. CategoryPanel) to enable plotting their distributions per selected category.


In [11]:
features = [
    cp_module.TargetFeature("energy_corrected_per_atom"),
    cp_module.TargetFeature("forces", lambda s: np.linalg.norm(np.vstack(s), axis=1)),
    cp_module.TargetFeature("mag_mom", lambda s: np.linalg.norm(np.vstack(s), axis=1)),
    cp_module.TargetFeature("nat"),
]


## 2.2. Category Panel

In [12]:
panel = cp_module.CategoryPanel(df, categories, features)
panel

CategoryPanel(children=(HBox(children=(VBox(children=(SelectMultiple(description='Categories:', layout=Layout(…

# 3. Plot Panel

In [13]:
filter_elements = [
    pp_module.make_slider(df, "energy_corrected_per_atom", "energy_corrected_per_atom", 0.01),
    pp_module.make_slider(df, "volume_per_atom", "volume_per_atom", 0.1),
    pp_module.make_slider(df, "force_max", "force_max", 0.01),
    pp_module.make_slider(df, "mag_mom_max", "mag_mom_max", 0.01),
    pp_module.make_slider(df, "M_abs_per_atom", "M_abs_per_atom", 0.01),
]

# this is a template - adjust to your interests
plot_funcs_dict = {
    "2D Scatter E vs V": lambda df: put.plot_xy_scatter(
        df, 
        x_col="volume_per_atom", 
        y_col="energy_corrected_per_atom"
    ),

    "2D Scatter E vs M_abs": lambda df: put.plot_xy_scatter(
        df, 
        x_col="M_abs_per_atom", 
        y_col="energy_corrected_per_atom"
    ),

    "2D Contour E(V,M_abs)": lambda df: put.plot_contour_interpolation(
        df, 
        x_col="volume_per_atom", 
        y_col="M_abs_per_atom", 
        z_col="energy_corrected_per_atom", 
        zlabel="Energy"
    ),

    "2D Contour E(V,nat)": lambda df: put.plot_contour_interpolation(
        df, 
        x_col="volume_per_atom", 
        y_col="nat", 
        z_col="energy_corrected_per_atom", 
        zlabel="Energy"
    ),
    
    "3D Scatter": lambda df: put.plot_3d_plotly(
        df, 
        x_col="volume_per_atom", 
        y_col="M_abs_per_atom", 
        z_col="energy_corrected_per_atom"
    ),
}


panel = pp_module.PlotPanel(
    df=df,
    categories=categories,
    plot_funcs=plot_funcs_dict,
    filter_elements=filter_elements,
)
panel

PlotPanel(children=(HBox(children=(HBox(children=(SelectMultiple(description='Categories:', layout=Layout(widt…

# 4. Knapsack panel

### 🎒 KnapsackPanel — Interactive Dataset Builder

The `KnapsackPanel` is an extension of the `PlotPanel` that allows you to **interactively construct a custom subset of structures** from your dataset.

#### Use case:
Suppose you're exploring the dataset using filters (energy, force, magnetism, etc.) and want to:
- 📌 **Save certain entries** that match your criteria,
- 🔄 **Add or remove** different sets of entries,
- 💾 Eventually **save** the constructed selection to file for later analysis.

#### How to use:
1. **Apply filters** using sliders and category selectors in the upper part.
2. Click **"Add"** to include the currently selected entries into the knapsack.
3. You can later:
   - Use **"Subtract"** to remove selected filtered entries from the knapsack (set substraction).
   - Use **"Reset"** to clear the knapsack entirely.
4. Use the **dropdown** to select how to visualize the contents of the knapsack.
5. Use the **SavePanel** to save your knapsack (e.g., as a `.pckl` file).

This enables you to create new datasets on the fly.

> Note: The knapsack uses the `name` column as a unique identifier when adding/removing.


In [16]:
k_panel = pp_module.KnapsackPanel(
    df=df,
    categories=categories,
    plot_funcs=plot_funcs_dict,
    filter_elements=filter_elements,
)
k_panel

KnapsackPanel(children=(HBox(children=(HBox(children=(SelectMultiple(description='Categories:', layout=Layout(…

In [21]:
pd.read_pickle('test.pickl').shape

(27181, 31)

In [ ]:
pd.read_pickle('test.pickl').columns